<a href="https://colab.research.google.com/github/gauravd12345/language_models/blob/main/seq2seq/seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [128]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

dataset = "hf://datasets/PaulineSanchez/Translation_words_and_sentences_english_french/data/train-00000-of-00001-3d14582ea46e1b17.parquet"
df = pd.read_parquet(dataset)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

!pip install sentencepiece
import sentencepiece as spm

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


device: cuda


In [129]:
c1, c2 = df.columns
enc = np.array(df[c1])
dec = np.array(df[c2])

for i in range(len(enc)):
  enc[i] = enc[i].lower()
  dec[i] = dec[i].lower()

for i in np.random.choice(np.arange(len(df)), 10):
  print(f"{enc[i]:<50} | {dec[i]}")

that doesn't change how i feel.                    | ça ne change pas ce que je ressens.
go away. i don't want to see you.                  | va-t'en ! je ne veux pas te voir.
do you mind if i sit here?                         | voyez-vous un inconvénient à ce que je m'asseye ici ?
the policemen fired at the car's tires.            | les policiers ont tiré sur les pneus de la voiture.
tell me what tom said.                             | dis-moi ce que tom a dit.
how'd you get that shiner?                         | comment as-tu pris ce maquereau ?
i have no choice but to take the red-eye back to new york. | je n'ai pas d'autre choix que de prendre le vol de nuit pour retourner à new-york.
we're not getting any younger.                     | nous ne rajeunissons pas.
what are friends for?                              | à quoi servent les amis ?
there are a lot of things i want to ask you.       | il y a beaucoup de choses que je souhaite vous demander.


In [130]:
with open("bpe_train.txt", "w", encoding="utf-8") as f:
    for s in enc:
        f.write(str(s).lower() + "\n")
    for s in dec:
        f.write(str(s).lower() + "\n")

spm.SentencePieceTrainer.train(
    input="bpe_train.txt",
    model_prefix="bpe",
    vocab_size=8000,
    model_type="bpe",
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

sp = spm.SentencePieceProcessor()
sp.load("bpe.model")

True

In [131]:
""" Hyperparameters """

embedding_dim = 300
hidden_size = 300 # context vector length
max_decoder_seq = 50
batch_size = 128

lr = 0.001
epochs = 10

pad_tok = sp.pad_id()
unk_tok = sp.unk_id()
sos_tok = sp.bos_id()
eos_tok = sp.eos_id()

vocab_len = sp.get_piece_size()
enc_vocab_len = vocab_len
dec_vocab_len = vocab_len

In [132]:
E = []
for sentence in enc:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    E.append(torch.tensor(ids, dtype=torch.long))

D = []
for sentence in dec:
    ids = sp.encode(str(sentence).lower(), out_type=int)
    ids = [sos_tok] + ids + [eos_tok]
    D.append(torch.tensor(ids, dtype=torch.long))

enc_pad_tok = pad_tok
dec_pad_tok = pad_tok

## RNN Encoder-Decoder

In [133]:
class Seq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Embedding(vocab_len, embedding_dim, padding_idx=pad_tok)
        self.dec_embed = nn.Embedding(vocab_len, embedding_dim, padding_idx=pad_tok)

        """ encoder RNN """
        self.E_x = nn.Linear(embedding_dim, hidden_size)
        self.E_h = nn.Linear(hidden_size, hidden_size)
        self.E_y = nn.Linear(hidden_size, hidden_size)

        """ decoder RNN """
        self.D_x = nn.Linear(embedding_dim, hidden_size)
        self.D_h = nn.Linear(hidden_size, hidden_size)
        self.D_y = nn.Linear(hidden_size, vocab_len)

    def forward(self, e, d=None):
        s = torch.zeros(e.size(0), hidden_size).to(device)  # initial hidden state s(0)

        w = self.enc_embed(e)
        for i in range(e.size(1)): # encoder pass
          w_i = w[:, i, :]
          s = torch.tanh(self.E_x(w_i) + self.E_h(s))
          y = self.E_y(s)

        out = []
        if d is not None: # training mode(teacher forcing)
          w = self.dec_embed(d)
          for i in range(d.size(1)):
            w_i = w[:, i, :]
            s = torch.tanh(self.D_x(w_i) + self.D_h(s))
            y = self.D_y(s)
            out.append(y)

        else:    # inference mode
          start = torch.full((e.size(0),), sos_tok, dtype=torch.long, device=device)
          w = self.dec_embed(start)
          for _ in range(max_decoder_seq):
            s = torch.tanh(self.D_x(w) + self.D_h(s))
            y = self.D_y(s)
            out.append(y)
            pred = torch.argmax(y, dim=1).item()

            w = self.dec_embed(torch.tensor(pred).to(device))

        return torch.stack(out, dim=1), s

seq2seq = Seq2Seq().to(device)

In [134]:
class Seq2SeqDataset(Dataset):
  def __init__(self, e, d):
    self.e = e
    self.d = d

  def __getitem__(self, index):
     return self.e[index], self.d[index]

  def __len__(self):
    return len(self.e)

""" collate function for batching """
def collate_fn(batch):
  e, d = zip(*batch)

  max_e = max(len(seq) for seq in e)
  max_d = max(len(seq) for seq in d)

  pad_e = torch.zeros(len(e), max_e, dtype=torch.long)
  pad_d = torch.zeros(len(d), max_d, dtype=torch.long)

  for i in range(len(e)):
    pad_e[i] = torch.cat([e[i], torch.tensor([enc_pad_tok] * (max_e - len(e[i])))])

  for i in range(len(d)):
    pad_d[i] = torch.cat([d[i], torch.tensor([dec_pad_tok] * (max_d - len(d[i])))])

  return pad_e, pad_d

### Training Loop

In [135]:
data_len = 50000
dataset = Seq2SeqDataset(E[:data_len], D[:data_len])
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

criterion = nn.CrossEntropyLoss(ignore_index=pad_tok)
optimizer = optim.Adam(seq2seq.parameters(), lr=lr)

for epoch in range(epochs):
  total_loss = 0.0
  for e, d in dataloader:
    e = e.to(device)
    d = d.to(device)

    optimizer.zero_grad()

    d_in, d_out = d[:, :-1], d[:, 1:]
    out, _ = seq2seq(e[:, :-1], d_in)

    loss = criterion(
        out.reshape(-1, out.size(-1)),
        d_out.reshape(-1)
    )

    total_loss += loss.item()

    loss.backward()
    optimizer.step()

  print(f"Epoch: {epoch+1}/{epochs} | loss: {total_loss / len(dataloader)}")

Epoch: 1/10 | loss: 3.8060291246380036
Epoch: 2/10 | loss: 2.900893009532138
Epoch: 3/10 | loss: 2.5950440884856008
Epoch: 4/10 | loss: 2.406345137549788
Epoch: 5/10 | loss: 2.2761400794739
Epoch: 6/10 | loss: 2.150376924468428
Epoch: 7/10 | loss: 1.9889815182941954
Epoch: 8/10 | loss: 1.878656776359929
Epoch: 9/10 | loss: 1.7932565343349487
Epoch: 10/10 | loss: 1.719462317883816


In [136]:
text = """
Yesterday I went to the market with my sister. We bought fresh bread, fruit, and some cheese for dinner.
The weather was cold, but the streets were full of people. After lunch, we visited a small bookstore near the river and
spent almost an hour looking at old novels. I was tired when I got home, but I still watched a movie before going to sleep.
"""

text = sent_tokenize(text)

for test in text:
  seq = sp.encode(test.lower(), out_type=int)
  seq = [sos_tok] + seq + [eos_tok]

  seq2seq.eval()

  with torch.no_grad():
      seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)
      pred, _ = seq2seq(seq)
      pred_ids = []

      for prob in pred[0]:
          pred_id = torch.argmax(prob).item()

          if pred_id == eos_tok:
              break

          if pred_id not in [sos_tok, pad_tok]:
              pred_ids.append(pred_id)

      print(sp.decode(pred_ids))

il est dans le garage.
nous ne pouvons pas nous aider.
tom est un homme marié.
tom est un homme marié.
je suis un étranger ici.


## GRU Encoder-Decoder

In [137]:
class GRUSeq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embed = nn.Embedding(vocab_len, embedding_dim, padding_idx=pad_tok)
        self.dec_embed = nn.Embedding(vocab_len, embedding_dim, padding_idx=pad_tok)

        """ encoder/decoder GRU """
        self.encoder = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.decoder = nn.GRU(embedding_dim, hidden_size, batch_first=True)

        self.fc = nn.Linear(hidden_size, vocab_len)

    def forward(self, e, d=None):
        s = torch.zeros(1, e.size(0), hidden_size, device=e.device)
        w = self.enc_embed(e)
        _, s = self.encoder(w, s)

        if d is None:
            tok = torch.full((e.size(0), 1), sos_tok, dtype=torch.long, device=e.device)
            outputs = []
            for _ in range(max_decoder_seq):
                w = self.dec_embed(tok)
                out, s = self.decoder(w, s)
                logits = self.fc(out)

                outputs.append(logits)
                tok = torch.argmax(logits[:, -1, :], dim=1).unsqueeze(1)

                if torch.all(tok.squeeze(1) == eos_tok):
                    break

            return torch.cat(outputs, dim=1)

        w = self.dec_embed(d)
        out, _ = self.decoder(w, s)
        out = self.fc(out)

        return out

gru_seq2seq = GRUSeq2Seq().to(device)

In [138]:
dataset = Seq2SeqDataset(E, D)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

criterion = nn.CrossEntropyLoss(ignore_index=dec_pad_tok)
optimizer = optim.Adam(gru_seq2seq.parameters(), lr=lr)

for epoch in range(epochs):
  total_loss = 0.0
  for e, d in dataloader:
    e = e.to(device)
    d = d.to(device)

    optimizer.zero_grad()

    d_in, d_out = d[:, :-1], d[:, 1:]
    out = gru_seq2seq(e[:, :-1], d_in)

    loss = criterion(
        out.reshape(-1, out.size(-1)),
        d_out.reshape(-1)
    )

    total_loss += loss.item()

    loss.backward()
    optimizer.step()

  print(f"Epoch: {epoch+1}/{epochs} | loss: {total_loss / len(dataloader)}")

Epoch: 1/10 | loss: 3.1081554371453306
Epoch: 2/10 | loss: 2.0379350414074398
Epoch: 3/10 | loss: 1.6701292677746473
Epoch: 4/10 | loss: 1.443290465683245
Epoch: 5/10 | loss: 1.2858053617978253
Epoch: 6/10 | loss: 1.1685234157383486
Epoch: 7/10 | loss: 1.0754596587083012
Epoch: 8/10 | loss: 1.0007128873765252
Epoch: 9/10 | loss: 0.938351477131541
Epoch: 10/10 | loss: 0.8849066617626243


In [147]:
text = """
Yesterday I went to the market with my sister. We bought fresh bread, fruit, and some cheese for dinner.
The weather was cold, but the streets were full of people. After lunch, we visited a small bookstore near the river and
spent almost an hour looking at old novels. I was tired when I got home, but I still watched a movie before going to sleep.
"""

text = sent_tokenize(text)

for test in text:
  seq = sp.encode(test.lower(), out_type=int)
  seq = [sos_tok] + seq + [eos_tok]

  gru_seq2seq.eval()

  with torch.no_grad():
      seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

      pred = gru_seq2seq(seq)

      pred_ids = []

      for prob in pred[0]:
          pred_id = torch.argmax(prob).item()

          if pred_id == eos_tok:
              break

          if pred_id not in [sos_tok, pad_tok]:
              pred_ids.append(pred_id)

      print(sp.decode(pred_ids))
  print()

hier j'ai rencontré par moi-même et j'ai trouvé un emploi.

nous avons acheté des fruits frais et burent du thé avec la cuisine.

le temps était étourdi que les murs étaient à traversés.

après avoir été diplômé de l'espion, j'ai pensé à un autre côté de notre part pendant le temps en retard au cours des états-unis.

j'étais alors je parlais à temps avant que je ne parte pas chez toi.

